In [ ]:
!pip install -q transformers sentence-transformers accelerate gradio pandas torch

In [ ]:
import pandas as pd
import torch
import gradio as gr

from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

In [ ]:
recipes = pd.read_csv("IndianFoodDatasetCSV.csv")

recipes = recipes.dropna(
    subset=[
        "TranslatedIngredients",
        "TranslatedInstructions"
    ]
)

recipes = recipes.reset_index(drop=True)

print("Total Recipes:", len(recipes))

Total Recipes: 6865


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

print("Qwen Loaded")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen Loaded


In [ ]:
def generate_recipe_ui(user_input):

    try:

        user_ingredients = [
            i.strip().lower()
            for i in user_input.split(",")
        ]

        def contains_ingredient(text):

            text = str(text).lower()

            return any(
                ing in text
                for ing in user_ingredients
            )

        filtered = recipes[
            recipes["TranslatedIngredients"].apply(
                contains_ingredient
            )
        ]

        if len(filtered) == 0:

            return "No matching recipes found."

        recipe_embeddings = embedding_model.encode(
            filtered["TranslatedIngredients"].tolist(),
            convert_to_tensor=True
        )

        user_embedding = embedding_model.encode(
            user_input,
            convert_to_tensor=True
        )

        scores = util.cos_sim(
            user_embedding,
            recipe_embeddings
        )[0]

        k = min(3, len(filtered))

        top_scores, top_indices = torch.topk(
            scores,
            k=k
        )

        context = ""

        for idx in top_indices:

            recipe = filtered.iloc[idx.item()]

            context += f"""
Recipe Name:
{recipe['TranslatedRecipeName']}

Ingredients:
{recipe['TranslatedIngredients']}

Instructions:
{recipe['TranslatedInstructions']}

----------------------------------
"""

        prompt = f"""
You are a professional chef.

Context:
{context}

Generate ONE recipe using:

{user_input}

FORMAT:

Recipe Name:
Ingredients:
Instructions:
Cooking Time:
Servings:
"""

        output = generator(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )

        generated_recipe = output[0]["generated_text"]

        # Evaluation

        generated_embedding = embedding_model.encode(
            generated_recipe,
            convert_to_tensor=True
        )

        similarity_scores = util.cos_sim(
            generated_embedding,
            recipe_embeddings[top_indices]
        )

        highest = similarity_scores.max().item()

        average = similarity_scores.mean().item()

        final_output = f"""
{generated_recipe}

----------------------------------

Evaluation

Highest Similarity: {highest:.4f}

Average Similarity: {average:.4f}
"""

        return final_output

    except Exception as e:

        return f"Error: {str(e)}"

In [ ]:
interface = gr.Interface(
    fn=generate_recipe_ui,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter ingredients separated by commas..."
    ),

    outputs=gr.Textbox(
        lines=25
    ),

    title="🍳 AI Recipe Generator",

    description="""
Generate recipes using Retrieval-Augmented Generation (RAG)
with Sentence Transformers and Qwen 2.5.
""",

    theme="soft"
)

In [ ]:
interface.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26062e4d8fbf6477a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
